# 14.2 - Prompt Versioning & A/B Testing

**Module 14 - LLMOps** | Netsetos GenAI Engineering

This notebook builds the whole prompt lifecycle as small runnable models: a versioned registry, a SemVer bumper, a deterministic traffic router, an A/B tally, a two-proportion significance test, a staged canary, and a rollback footprint tracker. Every cell runs keyless - each verdict or model call is replaced by a deterministic stand-in so you can read the mechanics end to end.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A one-line reproducibility note - the exercises lean on hashing and seeded values to stay deterministic, so every run of this notebook prints the same numbers.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a comment-only setup cell. There are no imports or installs to run here; each later cell imports the one thing it needs (`hashlib` or `math`) at the point of use, and everything runs on the Python standard library with no API key.

**How the code works - step by step**
- **The comment** - flags that the notebook uses seeded / deterministic values throughout, so outputs are stable across runs.

**In one line:** nothing executes - it just sets the expectation that results are reproducible.

**What you'll see:** (no output - environment prep)

## 1 - Store the prompt in a versioned registry

When you do not train the model, the prompt *is* the model, so it deserves a permanent address instead of living as a triple-quoted string in `main.py`. Here you build a tiny registry keyed by (id, semantic version), load two versions of the same classifier, and render each with a variable. This is the whole idea behind YAML-in-git prompt files: load by version, and the file is the contract.

In [ ]:
# The prompt IS the model when you do not train, so it needs a REGISTRY: every version has a permanent address
# (id + semver), stored as data (YAML in git), loaded by version, rendered with variables. The file is the contract.
REGISTRY = {   # in a real app these are prompts/<id>-v<version>.yaml files in git; here, one dict for a keyless demo
    ("review-classifier", "1.0.0"): {"model": "claude-sonnet-4-6", "temperature": 0.0, "max_tokens": 100,
        "template": "Classify this review into refund/delivery/food/service/other.\nReply with ONLY the label.\nReview: {review}",
        "eval_set": "golden/review-classifier.jsonl", "expected_pass_rate": 0.92},
    ("review-classifier", "1.1.0"): {"model": "claude-sonnet-4-6", "temperature": 0.0, "max_tokens": 100,
        "template": "Classify this review into refund/delivery/food/service/other.\nExample: 'cold pizza, want money back' -> refund\nReply with ONLY the label.\nReview: {review}",
        "eval_set": "golden/review-classifier.jsonl", "expected_pass_rate": 0.95},
}
def load_prompt(pid, version):
    key = (pid, version)
    if key not in REGISTRY:
        raise KeyError("no such prompt version: {} v{}".format(pid, version))
    return REGISTRY[key]
def render(spec, **kw):
    out = spec["template"]
    for k, v in kw.items():
        out = out.replace("{" + k + "}", v)
    return out
p10 = load_prompt("review-classifier", "1.0.0")
p11 = load_prompt("review-classifier", "1.1.0")
print("Loaded review-classifier v1.0.0: model={}, temp={}, eval bar={}".format(p10["model"], p10["temperature"], p10["expected_pass_rate"]))
print("Loaded review-classifier v1.1.0: model={}, temp={}, eval bar={}".format(p11["model"], p11["temperature"], p11["expected_pass_rate"]))
print("v1.1.0 adds a few-shot example -> its rendered prompt is {} chars vs v1.0.0's {} chars.".format(
      len(render(p11, review="pizza was cold")), len(render(p10, review="pizza was cold"))))
print("Both versions still exist and are loadable by address; a published version is NEVER edited in place.")
print("A hardcoded string in main.py has exactly one mutable state and no history - the registry is the fix.")

# Output:
# Loaded review-classifier v1.0.0: model=claude-sonnet-4-6, temp=0.0, eval bar=0.92
# Loaded review-classifier v1.1.0: model=claude-sonnet-4-6, temp=0.0, eval bar=0.95
# v1.1.0 adds a few-shot example -> its rendered prompt is 160 chars vs v1.0.0's 111 chars.
# Both versions still exist and are loadable by address; a published version is NEVER edited in place.
# A hardcoded string in main.py has exactly one mutable state and no history - the registry is the fix.

**Explanation**

The cell is a miniature prompt store: `REGISTRY` is a dict standing in for per-version YAML files, and two helpers read from it. `load_prompt` fetches a spec by its (id, version) address or raises if it does not exist; `render` fills the `{review}` placeholder. The point it demonstrates is that both versions coexist and stay independently loadable - v1.1.0 is a new, longer version, not an edit of v1.0.0.

**How the code works - step by step**
- **`REGISTRY`** - a dict mapping (prompt id, semver) to a full spec: model, temperature, max_tokens, the template, the eval set, and the expected pass rate. This is the contract each version carries.
- **`load_prompt`** - looks up the (id, version) key and returns the spec, raising `KeyError` for an unknown version - the permanent-address lookup.
- **`render`** - a plain string-replace of `{review}` (a stand-in for a real templating engine like Jinja with strict-undefined).
- **the prints** - load v1.0.0 and v1.1.0, report each one's model / temp / eval bar, and compare rendered lengths to show v1.1.0 is longer because it adds a few-shot example.

**In one line:** every version has an address, the file is the contract, and a published version is never edited in place.

**What you'll see:** It loads both versions (eval bars 0.92 and 0.95), reports that v1.1.0's rendered prompt is 160 chars vs v1.0.0's 111 chars because of the added few-shot example, and prints that both versions remain loadable while a hardcoded string has one mutable state and no history.

## 2 - Encode change risk with semantic versioning

A version number is only useful if it means something. This cell reads the MAJOR.MINOR.PATCH triple as a risk signal and walks a real change history through a `bump` function - so a typo, an added example, and an output-format switch each land on the right kind of bump.

In [ ]:
# Semantic versioning for prompts: the bump number encodes the RISK. PATCH = no behaviour change; MINOR = a
# backwards-compatible improvement; MAJOR = a breaking change (output shape/categories) that needs downstream updates.
RULES = {"patch": "no behaviour change; pass-rate must not drop (typo/whitespace)",
         "minor": "backwards-compatible improvement; pass-rate improves or stays flat",
         "major": "BREAKING output/behaviour change; NEW eval set + update downstream code in the same PR"}
def bump(version, part):
    major, minor, patch = (int(x) for x in version.split("."))
    if part == "major": major, minor, patch = major + 1, 0, 0
    elif part == "minor": minor, patch = minor + 1, 0
    else: patch += 1
    return "{}.{}.{}".format(major, minor, patch)
CHANGES = [("tighten wording 'only' -> 'ONLY'", "patch"),
           ("add 3 few-shot examples for ambiguous reviews", "minor"),
           ("switch plain-text output to JSON {category, confidence}", "major"),
           ("fix a stray trailing space", "patch")]
v = "1.0.0"
print("start        v{}".format(v))
for desc, part in CHANGES:
    v = bump(v, part)
    print("{:<6} -> v{:<7} {:<48} [{}]".format(part.upper(), v, desc, RULES[part].split(";")[0]))
print()
print("The MAJOR bump (v2.0.0) is the dangerous one: {}".format(RULES["major"]))
print("Two hard rules: never edit a published version in place, and log the prompt_version on every call.")

# Output:
# start        v1.0.0
# PATCH  -> v1.0.1   tighten wording 'only' -> 'ONLY'                 [no behaviour change]
# MINOR  -> v1.1.0   add 3 few-shot examples for ambiguous reviews    [backwards-compatible improvement]
# MAJOR  -> v2.0.0   switch plain-text output to JSON {category, confidence} [BREAKING output/behaviour change]
# PATCH  -> v2.0.1   fix a stray trailing space                       [no behaviour change]
#
# The MAJOR bump (v2.0.0) is the dangerous one: BREAKING output/behaviour change; NEW eval set + update downstream code in the same PR
# Two hard rules: never edit a published version in place, and log the prompt_version on every call.

**Explanation**

This is a rules-plus-simulator cell, not a model call. `RULES` states what each bump level promises, and `bump` does the arithmetic - resetting lower components as the level rises. Feeding it a list of realistic prompt edits shows the version number climbing 1.0.0 -> 1.0.1 -> 1.1.0 -> 2.0.0 -> 2.0.1 and highlights that the MAJOR jump is the one carrying real danger.

**How the code works - step by step**
- **`RULES`** - maps `patch` / `minor` / `major` to their meaning: no behaviour change, backwards-compatible improvement, and breaking change (new eval set + downstream code in the same PR).
- **`bump`** - splits the version into major/minor/patch ints and, per the requested level, increments that part while zeroing the ones below it.
- **`CHANGES`** - a list of four real edits paired with their correct bump level (wording tweak = patch, few-shot examples = minor, plain-text-to-JSON = major, stray space = patch).
- **the loop** - applies each bump in turn and prints the advancing version with its rule.

**In one line:** classify the edit's risk, and the bump level follows.

**What you'll see:** A history table walking v1.0.0 through v1.0.1, v1.1.0, v2.0.0, and v2.0.1, then a callout that the MAJOR bump (JSON output) needs a new eval set plus a downstream code change in the same PR, plus the two hard rules (never edit in place, always log the version).

## 3 - Route A/B traffic with a deterministic hash

To compare two versions you split users between them - but the split must be *deterministic*, so a given user always lands in the same arm. This cell hashes the user id to a stable number in [0,1) and assigns the arm by where it falls, then proves the same user maps to the same arm every time.

In [ ]:
# A/B routing needs DETERMINISTIC bucketing: hash the user id to a stable number in [0,1). The SAME user always
# lands in the SAME arm - across requests, servers, and restarts - so no user ever sees both arms and pollutes the test.
import hashlib
def bucket(user_id, experiment="review-classifier-ab"):
    h = int(hashlib.md5((experiment + "::" + user_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000            # stable, uniform-ish in [0.0, 1.0)
CANDIDATE_SHARE = 0.10                        # send 10% to the candidate
def arm(user_id):
    return "candidate" if bucket(user_id) < CANDIDATE_SHARE else "champion"
users = ["u{:05d}".format(i) for i in range(2000)]
cand = sum(1 for u in users if arm(u) == "candidate")
print("Routing {} users with a {:.0%} candidate share:".format(len(users), CANDIDATE_SHARE))
print("  candidate arm: {} users ({:.1%})   champion arm: {} users ({:.1%})".format(cand, cand / len(users), len(users) - cand, 1 - cand / len(users)))
print("  the split lands near the {:.0%} target from the hash alone - no stored user->arm table needed.".format(CANDIDATE_SHARE))
print()
print("Determinism check: user 'u00042' -> arm '{}' on every call (bucket {:.4f}).".format(arm("u00042"), bucket("u00042")))
print("  ...checked twice, same result: '{}' and '{}'.".format(arm("u00042"), arm("u00042")))
print("random.random() would reassign that user every request, so one person would see BOTH arms - test polluted.")

# Output:
# Routing 2000 users with a 10% candidate share:
#   candidate arm: 208 users (10.4%)   champion arm: 1792 users (89.6%)
#   the split lands near the 10% target from the hash alone - no stored user->arm table needed.
#
# Determinism check: user 'u00042' -> arm 'champion' on every call (bucket 0.5015).
#   ...checked twice, same result: 'champion' and 'champion'.
# random.random() would reassign that user every request, so one person would see BOTH arms - test polluted.

**Explanation**

The cell is a pure-function router. `bucket` hashes `experiment::user_id` with MD5 and reduces it to a fraction in [0,1); `arm` compares that fraction to the candidate share. Because it is a pure function of the id, no stored assignment table is needed - and running 2000 users through it shows the split lands near the target while a single user stays put across repeated calls.

**How the code works - step by step**
- **`bucket`** - MD5-hashes the experiment name plus user id, takes it mod 10,000, and divides to get a stable, roughly-uniform value in [0.0, 1.0).
- **`CANDIDATE_SHARE` / `arm`** - anyone whose bucket is below 0.10 goes to the candidate, everyone else to the champion - a 10% slice.
- **the split check** - routes 2000 synthetic users and counts each arm to show the observed share lands near the 10% target from the hash alone.
- **the determinism check** - calls `arm("u00042")` twice and prints the same result, contrasting it with `random.random()`, which would reassign the user every request.

**In one line:** hash the id, and the arm falls out the same way forever - no table, no leakage.

**What you'll see:** It routes 2000 users into ~208 candidate (10.4%) and 1792 champion (89.6%), then shows user 'u00042' maps to 'champion' (bucket 0.5015) on repeated calls, warning that `random` would put one person in both arms and pollute the test.

## 4 - Run the A/B test and read each arm

Now put routing and evaluation together: route each user by bucket, serve that arm's prompt, log the result with its version, then group the log per version to get each arm's pass rate. The catch unique to LLMs is that a 'pass' is a judge's verdict on open-ended prose, not an exact match.

In [ ]:
# The A/B test itself: route each user by the deterministic bucket, serve that arm's prompt, log the result per
# version. Grouping by prompt_version gives the per-arm pass rate. (Pass/fail here is a deterministic stand-in for a judge.)
import hashlib
def bucket(user_id, experiment="review-classifier-ab"):
    return (int(hashlib.md5((experiment + "::" + user_id).encode()).hexdigest(), 16) % 10_000) / 10_000
def arm(user_id):
    return "candidate" if bucket(user_id) < 0.10 else "champion"
TRUE_RATE = {"champion": 0.92, "candidate": 0.95}     # the (unknown-in-real-life) quality of each prompt
def judged_pass(user_id, rate):                        # deterministic stand-in for an LLM-as-judge verdict
    h = int(hashlib.md5(("verdict::" + user_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000 < rate
users = ["u{:05d}".format(i) for i in range(2000)]
tally = {"champion": [0, 0], "candidate": [0, 0]}      # [passes, total]
for u in users:
    a = arm(u); tally[a][1] += 1
    if judged_pass(u, TRUE_RATE[a]): tally[a][0] += 1
print("Per-arm results (group the trace log by prompt_version):")
for a in ("champion", "candidate"):
    passes, total = tally[a]
    print("  {:<9} n={:<5} passes={:<5} observed pass-rate={:.1%}".format(a, total, passes, passes / total))
print()
print("The candidate looks better here, but the candidate arm only saw {} requests.".format(tally["candidate"][1]))
print("Is the gap real, or noise? A point estimate is not enough - you test it for significance next.")

# Output:
# Per-arm results (group the trace log by prompt_version):
#   champion  n=1792  passes=1653  observed pass-rate=92.2%
#   candidate n=208   passes=196   observed pass-rate=94.2%
#
# The candidate looks better here, but the candidate arm only saw 208 requests.
# Is the gap real, or noise? A point estimate is not enough - you test it for significance next.

**Explanation**

This cell simulates one full experiment. It reuses the deterministic `bucket`/`arm` router, assigns each arm a hidden 'true' quality, and uses `judged_pass` - a deterministic stand-in for an LLM-as-judge - to decide pass/fail per user. The tally, grouped by arm, is exactly what you would get by grouping a real trace log on `prompt_version`; it deliberately leaves the candidate on a small sample to set up the significance question.

**How the code works - step by step**
- **`bucket` / `arm`** - the same deterministic router from cell 3, sending 10% to the candidate.
- **`TRUE_RATE`** - the (in real life unknown) underlying quality of each prompt: champion 0.92, candidate 0.95.
- **`judged_pass`** - hashes a per-user 'verdict' value and passes if it is under that arm's true rate - a deterministic stand-in for a real judge so the cell runs keyless.
- **the loop / `tally`** - routes 2000 users, records [passes, total] per arm, then prints the observed pass rate for each.

**In one line:** route -> serve -> log the version -> group the log to get each arm's pass rate.

**What you'll see:** Per-arm results: champion n=1792 at 92.2% and candidate n=208 at 94.2%, followed by the caution that the candidate's better rate rests on only 208 requests - so you cannot yet tell whether the gap is real or noise.

## 5 - Test whether the gap is real or noise

A lead on a small sample is usually luck. This cell runs a two-proportion z-test to ask whether an observed gap could have appeared by chance, and shows the counter-intuitive punchline: the *same* +3-point lift is inconclusive at n=100 per arm but a clear win at n=10,000 - sample size decides, not the gap.

In [ ]:
# Do NOT ship on a point estimate. A two-proportion z-test asks: could this gap be noise? Same +3pp lift is
# NOT significant at n=100 per arm but IS at n=10000 per arm. Sample size, not the gap, decides. (Callback: Lesson 1.8.)
import math
def two_prop_p(p1, n1, p2, n2):
    x1, x2 = p1 * n1, p2 * n2
    pool = (x1 + x2) / (n1 + n2)
    se = math.sqrt(pool * (1 - pool) * (1 / n1 + 1 / n2))
    if se == 0: return 0.0, 999.0
    z = (p2 - p1) / se
    p_two_sided = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))   # normal CDF via erf, no scipy
    return z, p_two_sided
for label, n in [("small test", 100), ("large test", 10_000)]:
    z, p = two_prop_p(0.92, n, 0.95, n)
    verdict = "SHIP (significant)" if p < 0.05 else "WAIT (could be noise)"
    print("{:<11} 92% vs 95% (+3pp), n={:>6}/arm -> z={:>5.2f}  p={:.4f}  -> {}".format(label, n, z, p, verdict))
print()
# how many samples per arm to detect a given lift around a 90% baseline (approx two-proportion sizing, alpha .05 / power .8)
def sample_size(p, lift, z_a=1.96, z_b=0.84):
    p2 = p + lift; pbar = (p + p2) / 2
    return math.ceil((z_a * math.sqrt(2 * pbar * (1 - pbar)) + z_b * math.sqrt(p * (1 - p) + p2 * (1 - p2))) ** 2 / lift ** 2)
for lift in (0.01, 0.03, 0.05):
    print("to detect a {:>2.0f}pp lift around a 90% baseline you need about {:>6} requests PER ARM.".format(lift * 100, sample_size(0.90, lift)))
print("Small lifts need huge samples. Low-traffic features: run longer, use a richer metric, or ship only large effects.")

# Output:
# small test  92% vs 95% (+3pp), n=   100/arm -> z= 0.86  p=0.3895  -> WAIT (could be noise)
# large test  92% vs 95% (+3pp), n= 10000/arm -> z= 8.60  p=0.0000  -> SHIP (significant)
#
# to detect a  1pp lift around a 90% baseline you need about  13480 requests PER ARM.
# to detect a  3pp lift around a 90% baseline you need about   1354 requests PER ARM.
# to detect a  5pp lift around a 90% baseline you need about    434 requests PER ARM.
# Small lifts need huge samples. Low-traffic features: run longer, use a richer metric, or ship only large effects.

**Explanation**

This is a pure statistics cell, no model involved. `two_prop_p` computes the pooled two-proportion z-statistic and turns it into a two-sided p-value using `math.erf` for the normal CDF (no scipy needed). A second helper, `sample_size`, inverts the question - how many samples per arm you need to detect a given lift - to show that small lifts demand large samples.

**How the code works - step by step**
- **`two_prop_p`** - pools the two arms' pass counts, computes the standard error and z-score, and converts z to a two-sided p-value via `math.erf` (normal CDF, no scipy).
- **the significance loop** - runs the identical 92% vs 95% comparison at n=100 and n=10,000 per arm and labels each SHIP (p<0.05) or WAIT.
- **`sample_size`** - an approximate power calculation (alpha .05, power .8) for the per-arm sample needed to detect a given lift around a 90% baseline.
- **the sizing loop** - prints the required sample for a 1pp, 3pp, and 5pp lift.

**In one line:** the same lift is noise on a small sample and a win on a large one - size, not the point estimate, decides.

**What you'll see:** The +3pp gap gives z=0.86, p=0.39 (WAIT) at n=100 but z=8.60, p=0.0000 (SHIP) at n=10,000; the sizing table shows a 1pp lift needs ~13,480, a 3pp lift ~1,354, and a 5pp lift ~434 requests per arm - so low-traffic features must run longer or demand large effects.

## 6 - Ramp a winner through a canary

Even a proven winner does not jump to 100%. This cell ramps a new version through 5 -> 25 -> 50 -> 100 percent, checking guardrail metrics at each stage against rollback triggers that *tighten* as more users are exposed - and it deliberately rolls back mid-ramp when a small dip trips a tight trigger.

In [ ]:
# A winning A/B does NOT go straight to 100%. Canary ramps 5 -> 25 -> 50 -> 100, and each stage has TIGHTER
# rollback triggers than the last (more users = less tolerance). A trigger trip auto-rolls-back. Shadow-test first for zero user risk.
STAGES = [(5, {"pass_drop": 2.0, "cost": 30, "p99": 2.0}),   # (traffic %, rollback-if thresholds)
          (25, {"pass_drop": 1.0, "cost": 15, "p99": 1.5}),
          (50, {"pass_drop": 0.5, "cost": 10, "p99": 1.5}),
          (100, None)]
OBSERVED = {5:  {"pass_drop": 0.1, "cost": 6,  "p99": 1.1},   # measured metrics at each stage (illustrative)
            25: {"pass_drop": 0.3, "cost": 9,  "p99": 1.2},
            50: {"pass_drop": 0.8, "cost": 9,  "p99": 1.3}}    # pass_drop 0.8 trips the tight 0.5 trigger at 50%
for pct, trig in STAGES:
    if trig is None:
        print("stage {:>3}%  -> FULL ROLLOUT reached".format(pct)); break
    m = OBSERVED[pct]
    tripped = [k for k in trig if m[k] > trig[k]]
    if tripped:
        print("stage {:>3}%  triggers {}  observed {}  -> ROLLBACK (tripped: {})".format(pct, trig, m, ", ".join(tripped))); break
    print("stage {:>3}%  observed pass_drop={}pp cost=+{}% p99={}x  -> all triggers pass, ADVANCE".format(pct, m["pass_drop"], m["cost"], m["p99"]))
print()
print("The 50% stage rolls back: a 0.8pp pass drop is fine at 5% (trigger 2.0) but trips the tighter 0.5 trigger at 50%.")
print("Modern order (2026): shadow (score both, serve neither) -> canary (1-5%, eval-gated) -> ramp. Rollback is automatic.")

# Output:
# stage   5%  observed pass_drop=0.1pp cost=+6% p99=1.1x  -> all triggers pass, ADVANCE
# stage  25%  observed pass_drop=0.3pp cost=+9% p99=1.2x  -> all triggers pass, ADVANCE
# stage  50%  triggers {'pass_drop': 0.5, 'cost': 10, 'p99': 1.5}  observed {'pass_drop': 0.8, 'cost': 9, 'p99': 1.3}  -> ROLLBACK (tripped: pass_drop)
#
# The 50% stage rolls back: a 0.8pp pass drop is fine at 5% (trigger 2.0) but trips the tighter 0.5 trigger at 50%.
# Modern order (2026): shadow (score both, serve neither) -> canary (1-5%, eval-gated) -> ramp. Rollback is automatic.

**Explanation**

The cell is a small state machine over rollout stages. `STAGES` pairs each traffic percentage with its rollback thresholds (which get stricter as traffic grows), and `OBSERVED` supplies illustrative measured metrics per stage. The loop advances stage by stage until either a threshold is exceeded (rollback) or 100% is reached - showing that the same 0.8pp dip that is fine early trips the tighter trigger at 50%.

**How the code works - step by step**
- **`STAGES`** - a list of (traffic %, trigger thresholds) with tightening `pass_drop` / `cost` / `p99` limits at each step; the final stage has no triggers (full rollout).
- **`OBSERVED`** - the measured metrics at each stage; note the 50% stage's 0.8pp pass drop is set to exceed the tight 0.5 trigger.
- **the loop** - at each stage compares observed against triggers, ADVANCEs if all pass, ROLLS BACK naming the tripped metric, or announces full rollout.

**In one line:** ramp with tightening triggers and auto-roll-back on the first trip; shadow-test first for zero user risk.

**What you'll see:** Stages 5% and 25% advance, then the 50% stage rolls back because its 0.8pp pass drop trips the tightened 0.5 trigger; a closing note gives the 2026 order - shadow (score both, serve neither) then canary then ramp, with automatic rollback.

## 7 - Understand what a rollback does *not* undo

Reverting the served version is the easy half and takes seconds to minutes. The hard truth this cell models: flipping the prompt back does not undo the side effects the bad version already produced - cached responses, database rows it wrote, and in-flight requests. The real fix is tagging every write with the version that made it.

In [ ]:
# Rollback speed is the EASY part. The hard part: reverting the prompt does NOT undo the side effects v2 already
# produced. Tag every cache entry and DB write with the prompt_version that made it, or you cannot find and heal them.
MECHANISMS = [("YAML in git, redeploy", 600), ("DB flag / active_version column", 30), ("prompt platform, one click", 10)]
print("Time to revert the SERVED version:")
for name, secs in MECHANISMS:
    print("  {:<34} ~{:>4} s".format(name, secs))
print()
AFTER_ROLLBACK = [("the served prompt version", "reverts immediately"),
                  ("responses cached from v2", "keep serving v2 output until the TTL expires"),
                  ("rows v2 wrote to your database", "stay v2-labelled forever unless you reprocess them"),
                  ("requests already in flight", "already shipped - you cannot recall them")]
print("What a rollback does NOT undo:")
for what, fate in AFTER_ROLLBACK:
    print("  - {:<32} {}".format(what, fate))
print()
print("So the fastest revert still leaves a v2 footprint in caches and data. The fix is not speed, it is TAGGING:")
print("stamp every cache key and every DB write with prompt_version, so a rollback can find and heal what v2 touched.")

# Output:
# Time to revert the SERVED version:
#   YAML in git, redeploy              ~ 600 s
#   DB flag / active_version column    ~  30 s
#   prompt platform, one click         ~  10 s
#
# What a rollback does NOT undo:
#   - the served prompt version        reverts immediately
#   - responses cached from v2         keep serving v2 output until the TTL expires
#   - rows v2 wrote to your database   stay v2-labelled forever unless you reprocess them
#   - requests already in flight       already shipped - you cannot recall them
#
# So the fastest revert still leaves a v2 footprint in caches and data. The fix is not speed, it is TAGGING:
# stamp every cache key and every DB write with prompt_version, so a rollback can find and heal what v2 touched.

**Explanation**

This is a reference / enumeration cell - no computation beyond printing two tables. `MECHANISMS` lists revert speeds by mechanism to make the point that speed is cheap, and `AFTER_ROLLBACK` enumerates the three footprints a revert leaves behind. Together they argue that the solution is not a faster revert but tagging every cache key and DB write with `prompt_version`.

**How the code works - step by step**
- **`MECHANISMS`** - three revert mechanisms with approximate times: YAML-in-git redeploy (~600s), DB flag (~30s), platform one-click (~10s).
- **the first loop** - prints each mechanism's revert time to show the served version reverts fast.
- **`AFTER_ROLLBACK`** - the four items and their fate: the served version reverts, but cached v2 responses persist to TTL, v2-written rows stay labelled, and in-flight requests already shipped.
- **the second loop** - prints each footprint, then the closing point: the fix is tagging every write with `prompt_version` so a rollback can find and heal what v2 touched.

**In one line:** a fast revert still leaves a v2 footprint in caches and data - tagging is what makes it findable.

**What you'll see:** A revert-time table (600s / 30s / 10s), then a list of what a rollback does not undo (cached responses until TTL, v2-labelled DB rows, in-flight requests), closing on the fix: stamp every cache key and DB write with the prompt_version.

Across seven cells you built the prompt as a first-class shipped artifact: a registry gives every version a permanent address, SemVer makes the bump encode risk, deterministic bucketing keeps each user in one arm, the A/B tally reads each arm's pass rate, the z-test tells you whether the gap is real or noise, the canary ramps a winner with tightening triggers, and the rollback model shows why tagging every write beats a faster revert. The through-line: never edit a published version, log the prompt_version on every call, and ship on evidence. Next, Lesson 14.3 builds the judge that defines "pass" - golden sets, LLM-as-judge, and the CI eval gate that actually declares an A/B winner.